
# Symmetry reduction benchmarks with `hamiltonian_triplet.py`

This notebook benchmarks the **triplet / COO -> CSR** Hamiltonian builder under four basis choices for the periodic Heisenberg chain:

- `FullBasis(L)`
- `SzBasis(L, L//2)`  (half filling)
- `MomentumBasis(L, m=0, n_up=None)`  (zero-momentum sector only)
- `MomentumBasis(L, m=0, n_up=L//2)`  (zero-momentum inside half filling)

The goal is to measure the **practical build-time improvement from symmetry reduction**, not just the Hilbert-space dimension reduction.

A useful warning up front: **smaller dimension does not automatically mean faster assembly**. Momentum sectors reduce the matrix size strongly, but each basis vector is itself a superposition over a translation orbit, so Hamiltonian construction has extra overhead.


In [ ]:

import time
from statistics import mean, stdev

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.sparse.linalg import eigsh

from basis import FullBasis, SzBasis, MomentumBasis
from hamiltonian_triplet import SpinChainHamiltonian as TripletHamiltonian

plt.rcParams["figure.figsize"] = (7, 4.5)


In [ ]:

def build_heisenberg_triplet(basis, Jxy=1.0, Jz=1.0, periodic=True):
    builder = TripletHamiltonian(basis)
    builder.add_heisenberg(Jxy=Jxy, Jz=Jz, periodic=periodic)
    return builder.build()


def benchmark_basis(factory, sizes, repeats=3, Jxy=1.0, Jz=1.0, periodic=True, label=None):
    rows = []
    for L in sizes:
        basis = factory(L)
        times = []
        nnz = None
        for _ in range(repeats):
            t0 = time.perf_counter()
            H = build_heisenberg_triplet(basis, Jxy=Jxy, Jz=Jz, periodic=periodic)
            times.append(time.perf_counter() - t0)
            nnz = H.nnz

        rows.append({
            "label": label,
            "L": L,
            "dim": basis.dim,
            "nnz": nnz,
            "time_mean": mean(times),
            "time_std": stdev(times) if len(times) > 1 else 0.0,
            "time_min": min(times),
            "time_max": max(times),
        })
    return pd.DataFrame(rows)


In [ ]:

# Quick sanity check: dimensions for the four basis choices at one size
L = 12
for name, basis in [
    ("Full", FullBasis(L)),
    ("Sz half filling", SzBasis(L, L//2)),
    ("Momentum only (m=0)", MomentumBasis(L, m=0, n_up=None)),
    ("Momentum + Sz (m=0, half filling)", MomentumBasis(L, m=0, n_up=L//2)),
]:
    H = build_heisenberg_triplet(basis)
    print(f"{name:30s}  dim={basis.dim:6d}  nnz={H.nnz:7d}")



## Benchmark plan

There are two complementary views:

1. **Common sizes** where all four basis choices are still feasible.
2. **Extended sizes** for the symmetry-reduced sectors, which can reach larger systems than the full basis.

The momentum benchmarks use **only the `m=0` sector**, so they represent the cost of building one translation-symmetry block, not the cost of reconstructing the full Hamiltonian from all momentum sectors.


In [ ]:

repeats = 3
common_sizes = [8, 10, 12]
extended_sz_sizes = [8, 10, 12, 14, 16]
extended_mom_sizes = [8, 10, 12, 14, 16]
extended_mom_sz_sizes = [8, 10, 12, 14, 16, 18]


In [ ]:

# Common-size comparison across all four basis choices

df_full = benchmark_basis(
    lambda L: FullBasis(L),
    common_sizes,
    repeats=repeats,
    label="FullBasis",
)

df_sz = benchmark_basis(
    lambda L: SzBasis(L, L // 2),
    common_sizes,
    repeats=repeats,
    label="Sz half filling",
)

df_mom = benchmark_basis(
    lambda L: MomentumBasis(L, m=0, n_up=None),
    common_sizes,
    repeats=repeats,
    label="Momentum only (m=0)",
)

df_mom_sz = benchmark_basis(
    lambda L: MomentumBasis(L, m=0, n_up=L // 2),
    common_sizes,
    repeats=repeats,
    label="Momentum + Sz (m=0, half filling)",
)

common_df = pd.concat([df_full, df_sz, df_mom, df_mom_sz], ignore_index=True)
common_df


In [ ]:

# Extended benchmarks for larger reachable sizes

df_sz_ext = benchmark_basis(
    lambda L: SzBasis(L, L // 2),
    extended_sz_sizes,
    repeats=repeats,
    label="Sz half filling",
)

df_mom_ext = benchmark_basis(
    lambda L: MomentumBasis(L, m=0, n_up=None),
    extended_mom_sizes,
    repeats=repeats,
    label="Momentum only (m=0)",
)

df_mom_sz_ext = benchmark_basis(
    lambda L: MomentumBasis(L, m=0, n_up=L // 2),
    extended_mom_sz_sizes,
    repeats=repeats,
    label="Momentum + Sz (m=0, half filling)",
)

extended_df = pd.concat([df_sz_ext, df_mom_ext, df_mom_sz_ext], ignore_index=True)
extended_df


In [ ]:

# Add reference full-basis dimensions on the same sizes

def full_dim(L):
    return 2 ** L

for frame in [common_df, extended_df]:
    frame["full_dim"] = frame["L"].map(full_dim)
    frame["dim_reduction_vs_full"] = frame["full_dim"] / frame["dim"]
    frame["nnz_per_dim"] = frame["nnz"] / frame["dim"]
    frame["time_per_dim"] = frame["time_mean"] / frame["dim"]
    frame["time_per_nnz"] = frame["time_mean"] / frame["nnz"]



## Common-size plots


In [ ]:

fig, ax = plt.subplots()
for label, sub in common_df.groupby("label"):
    ax.errorbar(sub["L"], sub["time_mean"], yerr=sub["time_std"], marker="o", capsize=3, label=label)
ax.set_xlabel("L")
ax.set_ylabel("build time [s]")
ax.set_title("Triplet builder: build time vs L")
ax.legend()
plt.show()


In [ ]:

fig, ax = plt.subplots()
for label, sub in common_df.groupby("label"):
    ax.plot(sub["dim"], sub["time_mean"], marker="o", label=label)
ax.set_xlabel("basis dimension")
ax.set_ylabel("build time [s]")
ax.set_title("Triplet builder: build time vs basis dimension")
ax.legend()
plt.show()


In [ ]:

pivot_speedup = common_df.pivot(index="L", columns="label", values="time_mean")
full_times = pivot_speedup["FullBasis"]

speedup_df = pd.DataFrame(index=pivot_speedup.index)
for col in pivot_speedup.columns:
    if col != "FullBasis":
        speedup_df[col] = full_times / pivot_speedup[col]

speedup_df


In [ ]:

fig, ax = plt.subplots()
for col in speedup_df.columns:
    ax.plot(speedup_df.index, speedup_df[col], marker="o", label=col)
ax.axhline(1.0, linestyle="--")
ax.set_xlabel("L")
ax.set_ylabel("speedup relative to FullBasis")
ax.set_title("Speedup from symmetry-reduced sectors")
ax.legend()
plt.show()


In [ ]:

fig, ax = plt.subplots()
for label, sub in common_df.groupby("label"):
    ax.plot(sub["L"], sub["dim_reduction_vs_full"], marker="o", label=label)
ax.set_xlabel("L")
ax.set_ylabel("dimension reduction factor vs full basis")
ax.set_title("Hilbert-space reduction")
ax.legend()
plt.show()


In [ ]:

fig, ax = plt.subplots()
for label, sub in common_df.groupby("label"):
    ax.plot(sub["L"], sub["time_per_nnz"], marker="o", label=label)
ax.set_xlabel("L")
ax.set_ylabel("time / nnz [s]")
ax.set_title("Assembly cost normalized by nnz")
ax.legend()
plt.show()



## Extended-size plots for symmetry-reduced sectors


In [ ]:

fig, ax = plt.subplots()
for label, sub in extended_df.groupby("label"):
    ax.errorbar(sub["L"], sub["time_mean"], yerr=sub["time_std"], marker="o", capsize=3, label=label)
ax.set_xlabel("L")
ax.set_ylabel("build time [s]")
ax.set_title("Extended benchmark: symmetry-reduced sectors")
ax.legend()
plt.show()


In [ ]:

fig, ax = plt.subplots()
for label, sub in extended_df.groupby("label"):
    ax.plot(sub["dim"], sub["time_mean"], marker="o", label=label)
ax.set_xlabel("basis dimension")
ax.set_ylabel("build time [s]")
ax.set_title("Extended benchmark: build time vs basis dimension")
ax.legend()
plt.show()


In [ ]:

fig, ax = plt.subplots()
for label, sub in extended_df.groupby("label"):
    ax.plot(sub["L"], sub["dim_reduction_vs_full"], marker="o", label=label)
ax.set_xlabel("L")
ax.set_ylabel("dimension reduction factor vs full basis")
ax.set_title("Extended benchmark: Hilbert-space reduction")
ax.legend()
plt.show()


In [ ]:

fig, ax = plt.subplots()
for label, sub in extended_df.groupby("label"):
    ax.plot(sub["L"], sub["time_per_nnz"], marker="o", label=label)
ax.set_xlabel("L")
ax.set_ylabel("time / nnz [s]")
ax.set_title("Extended benchmark: normalized assembly cost")
ax.legend()
plt.show()



## Optional eigensolver sanity check

This does **not** compare equal spectra across different symmetry sectors. It just checks that the constructed matrices are numerically usable by a sparse eigensolver.


In [ ]:

for name, basis in [
    ("Full", FullBasis(10)),
    ("Sz half filling", SzBasis(14, 7)),
    ("Momentum only", MomentumBasis(12, m=0, n_up=None)),
    ("Momentum + Sz", MomentumBasis(16, m=0, n_up=8)),
]:
    H = build_heisenberg_triplet(basis)
    vals = eigsh(H, k=2, which="SA", return_eigenvectors=False)
    print(f"{name:18s} dim={basis.dim:6d}  ground-ish eigenvalues = {vals}")



## Interpretation guide

A typical outcome for this code structure is:

- **Half-filled `SzBasis`** gives a large improvement, because the basis vectors are still single computational states.
- **Momentum only** often reduces the Hilbert-space dimension strongly, but may **not** improve assembly time as much as expected, because each momentum basis vector expands into an entire translation orbit.
- **Momentum + `Sz`** is often the strongest overall reduction, and may become the best option once the full and plain-`Sz` sectors get large enough.

So this notebook is useful precisely because it separates:

1. **dimension reduction**, and
2. **actual build-time reduction**.
